**TCGA PRC2 Loss-of-Function Data Discovery**

This notebook downloads TCGA masked somatic mutation data using TCGAbiolinks and identifies likely loss-of-function alterations in core PRC2 genes.

In [ ]:
# User settings

projects <- c("TCGA-LGG", "TCGA-GBM")

prc2_core <- c("EZH2", "EED", "SUZ12", "RBBP4", "RBBP7")

lof_classes <- c(
  "Nonsense_Mutation",
  "Frame_Shift_Del",
  "Frame_Shift_Ins",
  "Splice_Site"
)

output_dir <- "outputs/tcga_prc2_lof"
gdc_dir <- "GDCdata"

download_rna <- FALSE
download_methylation <- FALSE

In [ ]:
# Install and load packages

cran_packages <- c("dplyr", "readr")
bioc_packages <- c("TCGAbiolinks", "SummarizedExperiment")

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

for (pkg in bioc_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    BiocManager::install(pkg, ask = FALSE, update = FALSE)
  }
}

library(TCGAbiolinks)
library(SummarizedExperiment)
library(dplyr)
library(readr)

dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(gdc_dir, recursive = TRUE, showWarnings = FALSE)

In [ ]:
# Download TCGA Mutation data

query_maf <- GDCquery(
  project = projects,
  data.category = "Simple Nucleotide Variation",
  data.type = "Masked Somatic Mutation"
)

GDCdownload(query_maf, method = "api", directory = gdc_dir)

maf <- GDCprepare(query_maf, directory = gdc_dir)

In [ ]:
# Identify PRC2 loss of function events

prc2_lof <- maf %>%
  filter(
    Hugo_Symbol %in% prc2_core,
    Variant_Classification %in% lof_classes
  )

prc2_lof_samples <- prc2_lof %>%
  transmute(
    sample_id = Tumor_Sample_Barcode,
    patient_id = substr(Tumor_Sample_Barcode, 1, 12),
    project = substr(Tumor_Sample_Barcode, 1, 7),
    gene = Hugo_Symbol,
    variant_classification = Variant_Classification,
    chromosome = Chromosome,
    start_position = Start_Position,
    reference_allele = Reference_Allele,
    tumour_allele = Tumor_Seq_Allele2
  ) %>%
  distinct()

write_csv(
  prc2_lof_samples,
  file.path(output_dir, "PRC2_core_LoF_samples.csv")
)

In [ ]:
# Summarise PRC2 loss events

prc2_lof_event_summary <- prc2_lof_samples %>%
  count(project, gene, variant_classification, name = "n_events") %>%
  arrange(project, gene, desc(n_events))

write_csv(
  prc2_lof_event_summary,
  file.path(output_dir, "PRC2_core_LoF_event_summary.csv")
)

patient_ids <- unique(prc2_lof_samples$patient_id)
sample_ids <- unique(prc2_lof_samples$sample_id)

message("PRC2 LoF events: ", nrow(prc2_lof_samples))
message("Unique tumour samples: ", length(sample_ids))
message("Unique patients: ", length(patient_ids))

In [ ]:
# Download clinical metadata

